# O+O at RHIC: what is selection bias, what is quenching

The main analysis of the O+O physics run (phase 3 of the tuning page's "Physics run plan").
The numbers come from `oo_analysis_rhic.py` (`results/oo_analysis_rhic.json`): all finished jobs,
i.e. phase 3, the 3b extensions of the 7–18 GeV windows, and the narrow 3e windows at 35–40 and
47–55 GeV. Sections 5b and 5c show what 3e and 3b changed. This notebook only plots and tabulates. **Model results, not vetted physics results.**

**The samples** (`oo_bias.py`, tune XMLs of js-contrib `oo_rhic_run` `073085b` and later):

| sample | what | role |
|---|---|---|
| `oo_vac_w*` | O+O, the jet in vacuum (`--vacuum-jet`: Matter in vacuum, no LBT; the medium is made) | selection only |
| `oo_med_w*` | O+O, the jet in the medium (Matter + LBT in MUSIC), the same seeds | selection × quenching |
| `pp_ref_w*` | p+p in the full chain (`config/pp200_isr_rhictune.xml`), jet in vacuum | the R_AA reference |
| `pp_med_w*` | p+p, the jet in p+p's own medium (3c) | consistent reference, Matter's QS check |
| `phys_mb` | O+O minimum bias, bulk hadrons only | class edges, ⟨N_coll⟩ per class |

Common settings: soft MPI (pT0Ref 1.48), C_v 0.25, the RHIC bulk tune; p̂T windows 5–80 GeV,
stitched by their cross sections; MUSIC + iSS in every job.

**How the conclusion is drawn.** Per centrality class c of an estimator (STAR's EPD, sPHENIX's MBD,
mid-rapidity, or the impact parameter as truth):
- **Selection bias** = R_AA^c of `oo_vac` (the jet saw no medium; ≠ 1 only because jet events are
  sorted differently from minimum bias).
- **Quenching in the classes** Q^c = R_AA^c(`oo_med`) / R_AA^c(`oo_vac`) (the same classes; the
  normalisations cancel).
- **Total** = R_AA^c(`oo_med`) = selection × quenching, the quantity to compare with data.
- **Geometry check**: Q in impact-parameter classes (Q_b) against Q in the measured classes
  (Q_mult): whether the class sorting distorts the quenching itself.
- **I_CP** (STAR): per-trigger yields, the same split into vacuum (selection) and medium (total).

**The medium's holes.** iColored hadronizes the recoils and, for holes it cannot close, a colour
partner with the hole's three-momentum, but drops the holes: the hadrons hold recoil + partner where
the balance is recoil − hole.
- **Jets**: subtracted at parton level, the holes (`partons/`, pstat −1) and the partner copies
  (`hadronization_input/`) inside the cone: sPHENIX full jets with their full four-momentum, STAR's
  charged jets (inclusive and recoil) with the charged fraction 2/3 of their transverse momentum.
  The unsubtracted spectra are kept (`*_raw`).
- **Hadron-level observables (the charged-hadron spectra, R_AA and R_CP, and the h–h I_CP) are not
  hole-subtracted.** They need `prod_OO_isr/negative_hadrons.py` (the holes and partners hadronized
  as negative hadrons, counted with weight −1), not yet run on the phase-3 medium files. Holes are
  soft, so the effect should be small above ≈ 4 GeV, but it is not quantified; until then these
  quenched hadron results carry that caveat.

Statistics: medium and vacuum share seeds but not events (LBT shifts the random stream), so every
ratio of two samples has independent errors. Matter's vacuum cutoff differs between the arms (QS 0.9
with the jet in the medium, Q0 2 in vacuum); its size comes from `pp_med`'s events without a medium
(section 6).

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

R = json.loads(Path(os.environ.get("OO_ANALYSIS_RHIC_JSON", "results/oo_analysis_rhic.json")).read_text())
E = {k: np.asarray(v) for k, v in R["edges"].items()}
print("inputs:", R["inputs"]["out"], "| minimum bias:", R["inputs"]["mb_root"], R["inputs"]["mb_events"], "events",
      "| INTERIM: p+p reference missing" if R["inputs"].get("pp_missing") else "")
for f, n in R["inputs"]["events"].items():
    print(f"  {f:7s} {n:8d} events")
OBS = {"had": ("charged hadrons, |η| < 1 (holes not subtracted)", "$p_T$ [GeV]", E["had"]),
       "sph_jet": ("sPHENIX full jets R = 0.4, |η| < 0.7 (holes + partners subtracted)", "$p_{T,jet}$ [GeV]", E["sph_jet"]),
       "ch_jet_R0.2": ("STAR charged jets R = 0.2 (2/3 of holes + partners subtracted)", "$p_{T,jet}^{ch}$ [GeV]", E["ch_jet"]),
       "ch_jet_R0.5": ("STAR charged jets R = 0.5 (2/3 of holes + partners subtracted)", "$p_{T,jet}^{ch}$ [GeV]", E["ch_jet"])}
EST = {"epd": "STAR EPD", "mbd": "sPHENIX MBD", "mid10": "mid-rapidity |η|<1", "b": "impact parameter (truth)"}
COLS = plt.cm.viridis(np.linspace(0, 0.9, 6))

def xc(e):
    return 0.5 * (e[1:] + e[:-1]), 0.5 * (e[1:] - e[:-1])

def plot_raa(ax, block, fam, edges, title, which="raa"):
    x, xe = xc(edges)
    for c, (lo, hi) in enumerate(block["classes"]):
        y = np.asarray(block[fam][which][c], float); e = np.asarray(block[fam]["err"][c], float)
        ax.errorbar(x + (c - 2.5) * 0.08 * xe, y, e, xe, fmt="o", ms=3, color=COLS[c], label=f"{lo}-{hi}%")
    ax.axhline(1, color="k", lw=0.8); ax.set_title(title, fontsize=9); ax.set_ylim(0, 2)

def shade_incomplete(ax, est, key, edges):
    # grey: bins the stitching marks incomplete (pTHat below the lowest window is not generated)
    comp = R["raa"].get(est, {}).get(key, {}).get("stitch", {}).get("complete")
    if comp and not all(comp):
        ax.axvspan(edges[0], edges[max(i for i, c in enumerate(comp) if not c) + 1], color="0.88", zorder=0, lw=0)

## 1. Selection bias: R_AA of the vacuum jets per class

The jet never sees the medium, so any deviation from 1 is the selection: jet events are sorted into
the classes differently from minimum-bias events (the hard-momentum subtraction, the MPI, the jet's
own hadrons in the estimator). The impact-parameter row is the truth check (no selection by
construction; it should be 1 within the statistics, apart from what the hard-momentum subtraction
does to the inclusive spectrum).

In [ ]:
PP_MISSING = R["inputs"].get("pp_missing", False)
if PP_MISSING:
    print("pp_ref has no finished files yet: R_AA (Figure 1, R_AA 0-100 %, I_AA) needs it; R_CP, Q and I_CP do not")
if R["raa"] and not PP_MISSING:
    fig, ax = plt.subplots(len(EST), len(OBS), figsize=(16, 3.2 * len(EST)), sharey=True)
    for i, est in enumerate(EST):
        for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
            blk = R["raa"][est][key]
            if "oo_vac" in blk:
                plot_raa(ax[i, j], blk, "oo_vac", edges, f"{EST[est]}:\n{lab}")
                shade_incomplete(ax[i, j], est, key, edges)
            ax[-1, j].set_xlabel(xl)
        ax[i, 0].set_ylabel("$R_{AA}^{vac}$ (selection)")
    ax[0, 0].legend(fontsize=7, ncol=2)
    fig.suptitle("Figure 1. Selection bias: R_AA of oo_vac per class against N_coll × pp_ref", fontsize=11)
    fig.tight_layout(); plt.show()
elif not PP_MISSING:
    print("no R_AA (needs oo_vac and pp_ref)")

## 2. Quenching in the classes, and against the true geometry

Q^c = R_AA^c(`oo_med`) / R_AA^c(`oo_vac`): the effect of the medium on the jets of class c, with the
selection divided out. Q_b (impact-parameter classes) is the quenching at the true geometry; Q_mult
(EPD, MBD) what the measured classes show.

In [ ]:
def q_block(est, key):
    blk = R["raa"][est][key]
    if "oo_med" not in blk or "oo_vac" not in blk:
        return None
    a, ea = np.asarray(blk["oo_med"]["raa"], float), np.asarray(blk["oo_med"]["err"], float)
    b, eb = np.asarray(blk["oo_vac"]["raa"], float), np.asarray(blk["oo_vac"]["err"], float)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = a / b
        return blk["classes"], q, np.abs(q) * np.sqrt((ea / a) ** 2 + (eb / b) ** 2)

if R["raa"]:
    fig, ax = plt.subplots(len(EST), len(OBS), figsize=(16, 3.2 * len(EST)), sharey=True)
    for i, est in enumerate(EST):
        for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
            qb = q_block(est, key)
            if qb is None:
                continue
            cl, q, e = qb; x, xe = xc(edges)
            for c, (lo, hi) in enumerate(cl):
                ax[i, j].errorbar(x + (c - 2.5) * 0.08 * xe, q[c], e[c], xe, fmt="o", ms=3, color=COLS[c], label=f"{lo}-{hi}%")
            ax[i, j].axhline(1, color="k", lw=0.8); ax[i, j].set_ylim(0.3, 1.5); ax[i, j].set_title(f"{EST[est]}:\n{lab}", fontsize=9)
            shade_incomplete(ax[i, j], est, key, edges)
            ax[-1, j].set_xlabel(xl)
        ax[i, 0].set_ylabel("Q = R_AA(med) / R_AA(vac)")
    ax[0, 0].legend(fontsize=7, ncol=2)
    fig.suptitle("Figure 2. Quenching per class (selection divided out); last row: the true geometry (Q_b)", fontsize=11)
    fig.tight_layout(); plt.show()

## 3. Total: R_AA and R_CP of the quenched jets (the quantities the experiments measure)

R_AA^c(`oo_med`) = selection × quenching. R_CP between a central and a peripheral class cancels the
p+p reference; sPHENIX quotes 0–20 / 60–80 % (MBD), STAR 0–10 / 40–60 % (EPD).

In [ ]:
PAIRS = {"mbd": [((0, 20), (60, 80)), ((0, 20), (40, 60))], "epd": [((0, 10), (40, 60)), ((0, 10), (60, 80))]}
def rcp(est, key, fam, a, b):
    blk = R["raa"][est][key]
    if fam not in blk:
        return None
    cl = [tuple(c) for c in blk["classes"]]
    ia, ib = cl.index(a), cl.index(b)
    y, e = np.asarray(blk[fam]["raa"], float), np.asarray(blk[fam]["err"], float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = y[ia] / y[ib]
        return r, np.abs(r) * np.sqrt((e[ia] / y[ia]) ** 2 + (e[ib] / y[ib]) ** 2)

if R["raa"]:
    fig, ax = plt.subplots(2, len(OBS), figsize=(16, 6.5), sharey=True)
    for i, est in enumerate(("mbd", "epd")):
        for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
            x, xe = xc(edges)
            for k, (a, b) in enumerate(PAIRS[est]):
                for fam, mk, col in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
                    r = rcp(est, key, fam, a, b)
                    if r is not None:
                        ax[i, j].errorbar(x + (k - 0.5) * 0.15 * xe, r[0], r[1], xe, fmt=mk, ms=3, color=col,
                                          alpha=1 if k == 0 else 0.45,
                                          label=f"{fam} {a[0]}-{a[1]}/{b[0]}-{b[1]}%")
            ax[i, j].axhline(1, color="k", lw=0.8); ax[i, j].set_ylim(0.3, 1.5); ax[i, j].set_title(f"{EST[est]}:\n{lab}", fontsize=9)
            shade_incomplete(ax[i, j], est, key, edges)
            ax[-1, j].set_xlabel(xl)
        ax[i, 0].set_ylabel("$R_{CP}$"); ax[i, 0].legend(fontsize=7)
    fig.suptitle("Figure 3. R_CP: vacuum (selection only, squares) and medium (selection × quenching, circles)", fontsize=11)
    fig.tight_layout(); plt.show()

### Against sPHENIX (preliminary)

sPHENIX's R = 0.4 jet R_CP and R_AA (sPH-CONF-JET-2026-02, MBD classes, 19–60 GeV) read from the
note's vector figures (`refs/sphenix_oo_jet`; preliminary; the lowest bins' statistical bars are
upper bounds). Model: the full jets with the holes + partners subtracted, vacuum (selection) and
medium (total).

In [ ]:
SPD = (R.get("data") or {}).get("sphenix")
if SPD and R["raa"]:
    pairs = (("rcp_vs_60_80", "red", (0, 20), (60, 80)), ("rcp_vs_60_80", "blue", (20, 40), (60, 80)),
             ("rcp_vs_60_80", "green", (40, 60), (60, 80)), ("rcp_vs_40_60", "red", (0, 20), (40, 60)),
             ("rcp_vs_40_60", "blue", (20, 40), (40, 60)))
    fig, ax = plt.subplots(1, 6, figsize=(20, 3.6), sharey=True)
    x, xe = xc(E["sph_jet"])
    for j, (key, col, a, b) in enumerate(pairs):
        rows = SPD[key][col]
        xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); wd = np.array([r_["hi"] - r_["lo"] for r_ in rows])
        v = np.array([r_["value"] for r_ in rows])
        ax[j].errorbar(xd, v, [[r_["stat_lo"] for r_ in rows], [r_["stat_hi"] for r_ in rows]], wd / 2, fmt="*", ms=8, color="k", label="sPHENIX prel.")
        for xx, w_, r_ in zip(xd, wd, rows):
            ax[j].add_patch(plt.Rectangle((xx - w_ / 2, r_["value"] - r_["sys_lo"]), w_, r_["sys_lo"] + r_["sys_hi"], color="k", alpha=0.15))
        for fam, mk, c_ in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
            r = rcp("mbd", "sph_jet", fam, a, b)
            if r is not None:
                ax[j].errorbar(x, r[0], r[1], xe, fmt=mk, ms=4, color=c_, label=fam)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0, 1.6); ax[j].set_xlabel("$p_{T,jet}$ [GeV]")
        ax[j].set_title(f"R_CP {a[0]}-{a[1]} / {b[0]}-{b[1]} % (MBD)", fontsize=9)
    rows = SPD["raa_0_100"]["black"]
    xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); wd = np.array([r_["hi"] - r_["lo"] for r_ in rows])
    ax[5].errorbar(xd, [r_["value"] for r_ in rows], [[r_["stat_lo"] for r_ in rows], [r_["stat_hi"] for r_ in rows]], wd / 2,
                   fmt="*", ms=8, color="k", label="sPHENIX prel.")
    for fam, mk, c_ in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
        blk = R["raa"]["mbd"]["sph_jet"]
        if fam in blk and not PP_MISSING:
            ax[5].errorbar(x, blk[fam]["raa_mb"], blk[fam]["err_mb"], xe, fmt=mk, ms=4, color=c_, label=fam)
    ax[5].axhline(1, color="k", lw=0.8); ax[5].set_title("R_AA 0-100 %", fontsize=9); ax[5].set_xlabel("$p_{T,jet}$ [GeV]")
    ax[0].set_ylabel("$R_{CP}$ / $R_{AA}$"); ax[0].legend(fontsize=8)
    fig.suptitle("Figure 3b. sPHENIX jets: selection (oo_vac) and total (oo_med) against the preliminary data", fontsize=11)
    fig.tight_layout(); plt.show()

### The medium's holes in the sPHENIX jets

The full jets subtract, at parton level, the four-momenta of the medium's holes (`partons/`, pstat −1)
and of their colour partners' copies (`hadronization_input/`, the hole's three-momentum) inside
ΔR < 0.4: iColored hadronizes recoil + partner, the correct balance is recoil − hole. No negative
hadronization is needed for a jet's pT (it is for hadron-level observables). Below: the quenched R_CP
with and without the subtraction (the vacuum arm has no holes).

In [ ]:
if R["raa"]:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    x, xe = xc(E["sph_jet"])
    for i, (est, (a, b)) in enumerate((("mbd", ((0, 20), (60, 80))), ("epd", ((0, 10), (40, 60))))):
        for key, mk, col, lab in (("sph_jet", "o", "C3", "holes + partners subtracted"), ("sph_jet_raw", "^", "0.5", "raw")):
            r = rcp(est, key, "oo_med", a, b)
            if r is not None:
                ax[i].errorbar(x, r[0], r[1], xe, fmt=mk, ms=4, color=col, label=lab)
        ax[i].axhline(1, color="k", lw=0.8); ax[i].set_ylim(0.3, 1.5); ax[i].set_xlabel("$p_{T,jet}$ [GeV]")
        ax[i].set_title(f"{EST[est]}: oo_med R_CP {a[0]}-{a[1]} / {b[0]}-{b[1]} %", fontsize=9)
    ax[0].set_ylabel("$R_{CP}$"); ax[0].legend(fontsize=8)
    fig.tight_layout(); plt.show()

### Against the models in sPHENIX's note: JEWEL, Angantyr, the nPDF baseline

The note (sPH-CONF-JET-2026-02, Figs. 6, 8–10) compares its jets with three calculations; their
curves are read from the note's vector figures like the data (`refs/sphenix_oo_jet`, key `theory`;
the bands are the model uncertainties as drawn).
- **JEWEL 2.4.0** (Figs. 6 right, 9, 10): default configuration, CT10 NLO, no nPDF; JEWEL's own
  Bjorken medium with T_i = 0.26 GeV in 0–20 % (from Habich–Nagle–Romatschke hydro), scaled with
  ⟨N_part⟩^(1/3) for the other classes; recoils off and on. The classes are set by geometry, so
  **JEWEL's R_CP is quenching only**, with no selection bias.
- **Angantyr** (Fig. 8): the centrality-bias factors on R_CP (no quenching; the approach of Park et
  al., arXiv:2507.03603). This is the counterpart of `oo_vac`'s R_CP (selection only), from a different
  mechanism (Angantyr's sub-collisions against our strings + ISR + MPI + hydro).
- **nPDF baseline** (Fig. 6 left): MadGraph5_aMC@NLO + Pythia8 with nNNPDF3.0, EPPS21, TUJU21
  (68 % CL; arXiv:2410.22405), no quenching. Our chain has no nPDF: `oo_vac`'s R_AA 0–100 % sits at 1
  by construction, and the baseline shows the size of what is missing.

Top row: the totals, as sPHENIX shows them. Bottom row, the split. Left: selection, `oo_vac` against
Angantyr. Middle and right: quenching only: `oo_med`/`oo_vac` (our R_CP with the selection divided
out) against JEWEL, and sPHENIX divided by Angantyr's bias (the note's own reading).

In [ ]:
TH = (SPD or {}).get("theory")
def th_curve(ax, rows, color, label, ls="-"):
    x = np.array([r_["x"] for r_ in rows])
    ax.fill_between(x, [r_["lo"] for r_ in rows], [r_["hi"] for r_ in rows], color=color, alpha=0.25, lw=0)
    if rows[0]["value"] is not None:
        ax.plot(x, [r_["value"] for r_ in rows], ls, color=color, lw=1.5, label=label)
    else:
        ax.fill_between([], [], [], color=color, alpha=0.25, label=label)
def data_pts(ax, rows, label="sPHENIX prel.", div=None):
    xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); wd = np.array([r_["hi"] - r_["lo"] for r_ in rows])
    d = np.ones_like(xd) if div is None else div(xd)
    v = np.array([r_["value"] for r_ in rows]) / d
    ax.errorbar(xd, v, [np.array([r_["stat_lo"] for r_ in rows]) / d, np.array([r_["stat_hi"] for r_ in rows]) / d], wd / 2,
                fmt="*", ms=8, color="k", label=label, zorder=5)
    for xx, w_, r_, dd in zip(xd, wd, rows, d):
        ax.add_patch(plt.Rectangle((xx - w_ / 2, (r_["value"] - r_["sys_lo"]) / dd), w_, (r_["sys_lo"] + r_["sys_hi"]) / dd,
                                   color="k", alpha=0.15))
def interp(rows):
    x = np.array([r_["x"] for r_ in rows]); y = np.array([r_["value"] for r_ in rows])
    return lambda u: np.interp(u, x, y)
JOFF, JON, ANG = "C2", "C1", "C4"
PANELS = {"0-20/60-80": ((0, 20), (60, 80), "rcp_vs_60_80", "red"), "0-20/40-60": ((0, 20), (40, 60), "rcp_vs_40_60", "red"),
          "20-40/60-80": ((20, 40), (60, 80), "rcp_vs_60_80", "blue")}

if TH and R["raa"]:
    x, xe = xc(E["sph_jet"]); m = x > 19
    fig, ax = plt.subplots(2, 3, figsize=(17, 8.5))
    # top: totals
    for j, cls in enumerate(("0-20/60-80", "0-20/40-60")):
        a, b, dk, col = PANELS[cls]
        data_pts(ax[0, j], SPD[dk][col])
        for fam, mk, c_ in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
            r = rcp("mbd", "sph_jet", fam, a, b)
            if r is not None:
                ax[0, j].errorbar(x[m], r[0][m], r[1][m], xe[m], fmt=mk, ms=4, color=c_, label=f"{fam} ({'selection' if fam == 'oo_vac' else 'total'})")
        th_curve(ax[0, j], TH[f"jewel_{dk}"]["recoils_off"][cls], JOFF, "JEWEL, recoils off")
        th_curve(ax[0, j], TH[f"jewel_{dk}"]["recoils_on"][cls], JON, "JEWEL, recoils on", "--")
        th_curve(ax[0, j], TH[f"angantyr_bias_{dk}"][cls], ANG, "Angantyr bias factor", ":")
        ax[0, j].set_title(f"R_CP {cls} % (MBD)", fontsize=9)
    data_pts(ax[0, 2], SPD["raa_0_100"]["black"])
    blk = R["raa"]["mbd"]["sph_jet"]
    for fam, mk, c_ in (("oo_vac", "s", "C0"), ("oo_med", "o", "C3")):
        if fam in blk and not PP_MISSING:
            ax[0, 2].errorbar(x[m], np.asarray(blk[fam]["raa_mb"])[m], np.asarray(blk[fam]["err_mb"])[m], xe[m], fmt=mk, ms=4, color=c_, label=fam)
    th_curve(ax[0, 2], TH["jewel_raa_0_100"]["recoils_off"], JOFF, "JEWEL, recoils off")
    th_curve(ax[0, 2], TH["jewel_raa_0_100"]["recoils_on"], JON, "JEWEL, recoils on", "--")
    for name, c_ in zip(("nNNPDF3.0", "EPPS21", "TUJU21"), ("C5", "C9", "C8")):
        th_curve(ax[0, 2], TH["npdf_raa_0_100"][name], c_, f"no quenching, {name}")
    ax[0, 2].set_title("R_AA 0-100 %", fontsize=9)
    # bottom left: selection only
    for cls, mk in (("0-20/60-80", "s"), ("0-20/40-60", "D"), ("20-40/60-80", "v")):
        a, b, dk, col = PANELS[cls]
        r = rcp("mbd", "sph_jet", "oo_vac", a, b)
        c_ = {"0-20/60-80": "C0", "0-20/40-60": "C9", "20-40/60-80": "C7"}[cls]
        if r is not None:
            ax[1, 0].errorbar(x[m], r[0][m], r[1][m], xe[m], fmt=mk, ms=4, color=c_, label=f"oo_vac {cls}")
        th_curve(ax[1, 0], TH[f"angantyr_bias_{dk}"][cls], c_, f"Angantyr {cls}", ":")
    ax[1, 0].set_title("selection only: oo_vac R_CP vs Angantyr's bias factors", fontsize=9)
    # bottom middle/right: quenching only
    for j, cls in ((1, "0-20/60-80"), (2, "0-20/40-60")):
        a, b, dk, col = PANELS[cls]
        rm, rv = rcp("mbd", "sph_jet", "oo_med", a, b), rcp("mbd", "sph_jet", "oo_vac", a, b)
        if rm is not None and rv is not None:
            q = rm[0] / rv[0]; qe = np.abs(q) * np.hypot(rm[1] / rm[0], rv[1] / rv[0])
            ax[1, j].errorbar(x[m], q[m], qe[m], xe[m], fmt="o", ms=4, color="C3", label="oo_med / oo_vac (R_CP)")
        data_pts(ax[1, j], SPD[dk][col], "sPHENIX / Angantyr bias", div=interp(TH[f"angantyr_bias_{dk}"][cls]))
        th_curve(ax[1, j], TH[f"jewel_{dk}"]["recoils_off"][cls], JOFF, "JEWEL, recoils off")
        th_curve(ax[1, j], TH[f"jewel_{dk}"]["recoils_on"][cls], JON, "JEWEL, recoils on", "--")
        ax[1, j].set_title(f"quenching only, R_CP {cls} %", fontsize=9)
    for a_ in ax.flat:
        a_.axhline(1, color="k", lw=0.8); a_.set_xlim(17, 62); a_.set_ylim(0.2, 1.4); a_.set_xlabel("$p_{T,jet}$ [GeV]")
        a_.legend(fontsize=6.5, loc="lower left")
    ax[0, 0].set_ylabel("$R_{CP}$, $R_{AA}$"); ax[1, 0].set_ylabel("selection / quenching factor")
    fig.suptitle("Figure 3c. sPHENIX jets: our split (oo_vac, oo_med) against the note's models (JEWEL, Angantyr, nPDF baseline)", fontsize=11)
    fig.tight_layout(); plt.show()

    print("at sPHENIX's bins (model curves interpolated at the bin centres):")
    for cls in ("0-20/60-80", "0-20/40-60", "20-40/60-80"):
        a, b, dk, col = PANELS[cls]
        rows = SPD[dk][col]; xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows])
        rv, rm = rcp("mbd", "sph_jet", "oo_vac", a, b), rcp("mbd", "sph_jet", "oo_med", a, b)
        print(f"  R_CP {cls}: bin  sPHENIX  oo_vac  Angantyr | oo_med  JEWEL-off JEWEL-on | oo_med/oo_vac  sPHENIX/Angantyr")
        for i, r_ in enumerate(rows):
            k = int(np.argmin(np.abs(E["sph_jet"][:-1] - r_["lo"])))
            k = k if abs(E["sph_jet"][k] - r_["lo"]) < 0.2 and abs(E["sph_jet"][k + 1] - r_["hi"]) < 0.2 else None
            ang = interp(TH[f"angantyr_bias_{dk}"][cls])(xd[i])
            jo, jn = interp(TH[f"jewel_{dk}"]["recoils_off"][cls])(xd[i]), interp(TH[f"jewel_{dk}"]["recoils_on"][cls])(xd[i])
            vv = f"{rv[0][k]:.2f}" if (k is not None and rv is not None) else "  - "
            mm = f"{rm[0][k]:.2f}" if (k is not None and rm is not None) else "  - "
            qq = f"{rm[0][k] / rv[0][k]:.2f}" if (k is not None and rm is not None and rv is not None) else "  - "
            print(f"    {r_['lo']:.0f}-{r_['hi']:.0f}  {r_['value']:.2f}     {vv}    {ang:.2f}    |  {mm}    {jo:.2f}      {jn:.2f}    |     {qq}          {r_['value'] / ang:.2f}")
elif not TH:
    print("no model curves: rerun refs/sphenix_oo_jet/extract_sphenix_oo_jet.py NOTE.pdf, then oo_analysis_rhic.py")

## 4. STAR's I_CP: h–h and h+jet

STAR's definitions (arXiv:2604.13935): one charged trigger 7–30 GeV (|η| < 1.5, chosen at random if
several); h–h associated 3–7 GeV, near |Δφ| < 0.8, recoil |Δφ − π| < 1, flat background from
0.8 < |Δφ| < 1.6 (STAR's v_n = 0 variation); h+jet recoil charged jets R = 0.2, 0.5, |Δφ − π| < π/4,
8–19 GeV. In the model the hard event alone is used (no ρA, no mixed events). The recoil charged
jets subtract the medium's holes and partner copies in the cone with the charged fraction 2/3 of
their transverse momentum (grey triangles: without); the trigger and the h–h associated hadrons are
hadron-level and not subtracted: the h–h I_CP needs `negative_hadrons.py` (see the note above). Per-trigger yields
are N_coll-free; the selection still enters through the trigger's class assignment. Points: the
model's vacuum (selection) and medium (total) arms; stars: STAR's 0–10 % values (text of the paper;
the full HEPData tables to be added).

In [ ]:
ic = R.get("icp") or {}
STD = (R.get("data") or {}).get("star", {})
def star_cols(tab):
    # {column name: (x labels, values array [value, stat, syst])}
    return {c["name"]: (tab["x"], np.asarray(c["values"], float)) for c in tab["cols"]}

if ic and ("oo_vac" in ic or "oo_med" in ic):
    cl = ic["classes"]; labs = [f"{a}-{b}%" for a, b in cl]
    print("windows used for the triggers:", ic.get("windows_used"))
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
    data_cols = {}
    if STD:
        hh, hj = star_cols(STD["icp_hh_vs_ea"]), star_cols(STD["icp_hjet_vs_ea"])
        data_cols = {"near": [v for k, v in hh.items() if "near" in k][0], "recoil": [v for k, v in hh.items() if "recoil" in k][0],
                     "hjet_R0.2_8_19": [v for k, v in hj.items() if "0.2" in k][0], "hjet_R0.5_8_19": [v for k, v in hj.items() if "0.5" in k][0]}
    for j, obs in enumerate(("near", "recoil", "hjet_R0.2_8_19", "hjet_R0.5_8_19")):
        for k, (fam, mk, col) in enumerate((("oo_vac", "s", "C0"), ("oo_med", "o", "C3"))):
            if fam in ic:
                v = np.asarray(ic[fam]["icp"][obs], float)
                ax[j].errorbar(np.arange(len(cl)) + (k - 0.5) * 0.15, v[:, 0], v[:, 1], fmt=mk, color=col, label=fam)
                raw = obs.replace("_8_19", "_raw_8_19")
                if fam == "oo_med" and raw != obs and raw in ic[fam]["icp"]:
                    v = np.asarray(ic[fam]["icp"][raw], float)
                    ax[j].errorbar(np.arange(len(cl)) + 0.2, v[:, 0], v[:, 1], fmt="^", color="0.6", ms=4,
                                   label="oo_med, holes not subtracted")
        if obs in data_cols:
            xl, vals = data_cols[obs]
            pos = [labs.index(x.replace("–", "-") + "%") for x in xl]
            ax[j].errorbar(np.asarray(pos) - 0.25, vals[:, 0], vals[:, 1], fmt="*", ms=10, color="k", label="STAR (stat)")
            for p_, v_ in zip(pos, vals):
                ax[j].add_patch(plt.Rectangle((p_ - 0.33, v_[0] - v_[2]), 0.16, 2 * v_[2], color="k", alpha=0.2))
        ax[j].set_xticks(range(len(cl))); ax[j].set_xticklabels(labs, rotation=45, fontsize=8)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0.4, 1.6); ax[j].set_title(obs, fontsize=9)
    ax[0].set_ylabel("$I_{CP}$ (class / 40-60 %)"); ax[1].legend(fontsize=8)
    fig.suptitle("Figure 4. I_CP in STAR's EPD classes: vacuum arm = selection, medium arm = total; STAR (HEPData; boxes: syst)", fontsize=11)
    fig.tight_layout(); plt.show()
    for fam in ("oo_vac", "oo_med"):
        if fam in ic:
            print(fam, "I_AA (0-100 % over p+p):", {k: tuple(round(x, 3) for x in v) for k, v in ic[fam]["iaa"].items()})
else:
    print("no I_CP yet")

**Recoil jets vs jet pT.** STAR's Fig. 6 (I_CP 0–10 / 40–60 % per jet-pT bin) and Fig. 5 (the
per-trigger recoil-jet spectra per class, $(1/N_{trig})\,d^2N/dp_T d\eta$, with
Δη = 2 (1.5 − R)). The model's spectra are the hard event's charged jets with 2/3 of the holes +
partners subtracted, no unfolding needed (no detector).

In [ ]:
he = E["hjet"]; x, xe = xc(he)
if ic and STD:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    pt = STD["icp_hjet_vs_pt"]
    for j, R_ in enumerate((0.2, 0.5)):
        c = [cc for cc in pt["cols"] if cc["qual"].get("RJET") == str(R_)][0]
        v = np.asarray(c["values"], float); lo_hi = np.asarray(pt["x"], float)
        xd = lo_hi.mean(1)
        ax[j].errorbar(xd, v[:, 0], v[:, 1], 0.5 * np.diff(lo_hi, axis=1).ravel(), fmt="*", ms=9, color="k", label="STAR")
        for xx, w_, vv in zip(xd, np.diff(lo_hi, axis=1).ravel(), v):
            ax[j].add_patch(plt.Rectangle((xx - w_ / 2, vv[0] - vv[2]), w_, 2 * vv[2], color="k", alpha=0.15))
        for k, (fam, mk, col) in enumerate((("oo_vac", "s", "C0"), ("oo_med", "o", "C3"))):
            if fam in ic:
                m = np.asarray(ic[fam][f"icp_vs_pt_hjet_R{R_}"], float)
                ax[j].errorbar(x + (k - 0.5) * 0.2, m[:, 0], m[:, 1], xe, fmt=mk, color=col, label=fam)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0.2, 1.6); ax[j].set_xlabel("$p_{T,jet}^{ch}$ [GeV]")
        ax[j].set_title(f"recoil-jet I_CP 0-10 / 40-60 %, R = {R_}", fontsize=9)
    ax[0].set_ylabel("$I_{CP}$"); ax[0].legend(fontsize=8)
    fig.tight_layout(); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    ccl = [tuple(c) for c in ic["classes"]]
    for j, R_ in enumerate((0.2, 0.5)):
        tab = STD[f"hjet_spectra_R{R_}"]; lo_hi = np.asarray(tab["x"], float); xd = lo_hi.mean(1)
        deta = 2 * (1.5 - R_)
        for k, c in enumerate(tab["cols"]):
            ea = c["qual"].get("EA interval").replace("–", "-").replace("%", ""); v = np.asarray(c["values"], float)
            ax[j].errorbar(xd, v[:, 0] * 10 ** (4 - k), v[:, 1] * 10 ** (4 - k), fmt="*", ms=7, color=COLS[k], label=f"STAR {ea}% x10^{4-k}")
            a, b = (int(t) for t in ea.split("-"))
            if "oo_med" in ic and (a, b) in ccl:
                y = np.asarray(ic["oo_med"]["yields"][ccl.index((a, b))][f"hjet_R{R_}"], float)
                ax[j].errorbar(x, y[:, 0] / (2 * xe) / deta * 10 ** (4 - k), y[:, 1] / (2 * xe) / deta * 10 ** (4 - k), xe,
                               fmt="o", mfc="none", ms=5, color=COLS[k])
        ax[j].set_yscale("log"); ax[j].set_xlabel("$p_{T,jet}^{ch}$ [GeV]"); ax[j].set_title(f"R = {R_}: STAR (stars) and oo_med (open circles)", fontsize=9)
    ax[0].set_ylabel("$(1/N_{trig})\\, d^2N/dp_T d\\eta$  [GeV$^{-1}$]"); ax[1].legend(fontsize=7)
    fig.tight_layout(); plt.show()

## 5. The p+p reference with the jet in its own medium (3c)

`pp_med` / `pp_ref`: the model's quenching in p+p together with Matter's lower vacuum cutoff (QS 0.9
GeV in the medium arm against Q0 2 GeV in the vacuum arm). Expected close to 1.

In [ ]:
pr = R.get("pp_med_over_ref")
if pr:
    fig, ax = plt.subplots(1, len(OBS), figsize=(16, 3.2), sharey=True)
    for j, (key, (lab, xl, edges)) in enumerate(OBS.items()):
        x, xe = xc(edges)
        ax[j].errorbar(x, pr[key]["ratio"], pr[key]["err"], xe, fmt="o", ms=3)
        ax[j].axhline(1, color="k", lw=0.8); ax[j].set_ylim(0.6, 1.4); ax[j].set_title(lab, fontsize=9); ax[j].set_xlabel(xl)
    ax[0].set_ylabel("pp_med / pp_ref")
    fig.tight_layout(); plt.show()
else:
    print("pp_med not run yet (3c)")

## 5b. What 3e changed: the high-p_T full jets with and without the narrow p̂_T windows

All figures above use phase 3 + 3b + 3e. 3e added narrow windows at 35–40 and 47–55 GeV (10k events per
arm and in p+p), where the wide windows had few events and set the errors of the sPHENIX bins across
the window edges. This section shows only the difference: phase 3 alone (`oo_analysis_rhic.py
--phase3`) against phase 3 + 3e (`--phase3 --with-3e`), the same events otherwise, neither with the
3b extensions. The data are in Figures 3b and 3c. The predicted errors are the estimate made before
3e ran (tuning page, "3e").

In [ ]:
P3, P3E = Path("results/oo_analysis_rhic_phase3.json"), Path("results/oo_analysis_rhic_phase3e.json")
if P3.exists() and P3E.exists():
    A3, A3E = json.loads(P3.read_text()), json.loads(P3E.read_text())
    Es = np.asarray(A3E["edges"]["sph_jet"]); x, xe = xc(Es); hi_ = Es[:-1] >= 19
    PRED = {"oo_vac": [0.05, 0.06, 0.05, 0.04, 0.06, 0.05], "oo_med": [0.05, 0.07, 0.05, 0.04, 0.04, 0.06]}
    PRED_RAA = [0.032, 0.034, 0.033, 0.024, 0.028, 0.027]
    def rcp_of(Rr, fam, a=(0, 20), b=(60, 80)):
        blk = Rr["raa"]["mbd"]["sph_jet"]; cl = [tuple(c) for c in blk["classes"]]
        y, e = np.asarray(blk[fam]["raa"]), np.asarray(blk[fam]["err"]); ia, ib = cl.index(a), cl.index(b)
        r = y[ia] / y[ib]; return r, r * np.hypot(e[ia] / y[ia], e[ib] / y[ib])
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
    for fam, c, mk in (("oo_vac", "C0", "s"), ("oo_med", "C3", "o")):
        r3, e3 = rcp_of(A3, fam); r3e, e3e = rcp_of(A3E, fam)
        ax[0].errorbar(x[hi_] - 0.4, r3[hi_], e3[hi_], fmt=mk, mfc="none", color=c, alpha=0.6, label=f"{fam}, phase 3")
        ax[0].errorbar(x[hi_] + 0.4, r3e[hi_], e3e[hi_], fmt=mk, color=c, label=f"{fam}, phase 3 + 3e")
        ax[1].plot(x[hi_], e3[hi_], mk + "--", mfc="none", color=c, label=f"{fam}, phase 3")
        ax[1].plot(x[hi_], e3e[hi_], mk + "-", color=c, label=f"{fam}, phase 3 + 3e")
        ax[1].plot(x[hi_], PRED[fam], "x", color=c, ms=8, label=f"{fam}, predicted")
        print(f"{fam} R_CP 0-20/60-80 (MBD), bins " + " ".join(f"{a:g}-{b:g}" for a, b in zip(Es[:-1][hi_], Es[1:][hi_])))
        print("   phase 3      : " + " ".join(f"{a:.2f}±{b:.2f}" for a, b in zip(r3[hi_], e3[hi_])))
        print("   phase 3 + 3e : " + " ".join(f"{a:.2f}±{b:.2f}" for a, b in zip(r3e[hi_], e3e[hi_])))
        print("   predicted err: " + " ".join(f"   ±{b:.2f}" for b in PRED[fam]))
    for Rr, lab, st, al in ((A3, "phase 3", "none", 0.6), (A3E, "phase 3 + 3e", None, 1.0)):
        blk = Rr["raa"]["mbd"]["sph_jet"]
        for fam, c, mk in (("oo_vac", "C0", "s"), ("oo_med", "C3", "o")):
            ax[2].errorbar(x[hi_] + (0.4 if st is None else -0.4), np.asarray(blk[fam]["raa_mb"])[hi_], np.asarray(blk[fam]["err_mb"])[hi_],
                           fmt=mk, mfc=st, color=c, alpha=al, label=f"{fam}, {lab}")
    e3m = np.asarray(A3["raa"]["mbd"]["sph_jet"]["oo_med"]["err_mb"]); e3em = np.asarray(A3E["raa"]["mbd"]["sph_jet"]["oo_med"]["err_mb"])
    print("oo_med R_AA 0-100 % error, phase 3 -> phase 3 + 3e (predicted): " +
          " ".join(f"{a:.3f}->{b:.3f} ({c:.3f})" for a, b, c in zip(e3m[hi_], e3em[hi_], PRED_RAA)))
    for a_, t_ in zip(ax, ("R_CP 0-20/60-80 % (MBD)", "R_CP statistical error", "R_AA 0-100 % (MBD)")):
        a_.set_xlabel("$p_{T,jet}$ [GeV]"); a_.set_title(t_, fontsize=9); a_.legend(fontsize=6.5)
    ax[0].axhline(1, color="k", lw=0.8); ax[2].axhline(1, color="k", lw=0.8); ax[0].set_ylim(0.2, 1.4); ax[2].set_ylim(0.4, 1.3)
    fig.suptitle("Figure 5b. sPHENIX jets, what 3e changed: phase 3 (open) against phase 3 + 3e (filled); x = predicted error", fontsize=11)
    fig.tight_layout(); plt.show()
    # quenching factor in impact-parameter classes, before / after
    for Rr, lab in ((A3, "phase 3     "), (A3E, "phase 3 + 3e")):
        bb = Rr["raa"]["b"]["sph_jet"]; m_, em = np.asarray(bb["oo_med"]["raa"])[0], np.asarray(bb["oo_med"]["err"])[0]
        v_, ev_ = np.asarray(bb["oo_vac"]["raa"])[0], np.asarray(bb["oo_vac"]["err"])[0]; q = m_ / v_; eq = q * np.hypot(em / m_, ev_ / v_)
        print(f"Q 0-20 % by b, {lab}: " + " ".join(f"{a:.2f}±{b:.2f}" for a, b in zip(q[hi_], eq[hi_])))
else:
    print("needs results/oo_analysis_rhic_phase3.json and _phase3e.json (oo_analysis_rhic.py --phase3 [--with-3e])")

## 5c. What 3b changed: the low windows (STAR's triggers, the hadrons, the low jet bins)

3b added events to the 7–9, 9–11, 11–14 and 14–18 GeV p̂_T windows of both O+O arms, where STAR's
7–30 GeV triggers and the low jet and hadron bins come from. Phase 3 + 3e (`--phase3 --with-3e`)
against all jobs (phase 3 + 3b + 3e); only the statistics differ.

In [ ]:
P3E, PALL = Path("results/oo_analysis_rhic_phase3e.json"), Path("results/oo_analysis_rhic.json")
if P3E.exists() and PALL.exists():
    A3E, AALL = json.loads(P3E.read_text()), json.loads(PALL.read_text())
    for Rr, lab in ((A3E, "phase 3 + 3e     "), (AALL, "phase 3 + 3b + 3e")):
        print(lab, {f: n for f, n in Rr["inputs"]["events"].items()})
    i3, ia = A3E["icp"], AALL["icp"]; cl = i3["classes"]; labs = [f"{a}-{b}%" for a, b in cl]
    obs_ = ("near", "recoil", "hjet_R0.2_8_19", "hjet_R0.5_8_19")
    fig, ax = plt.subplots(2, 4, figsize=(17, 6.4), sharex=True)
    for j, obs in enumerate(obs_):
        for k, (fam, mk, col) in enumerate((("oo_vac", "s", "C0"), ("oo_med", "o", "C3"))):
            v3, va = np.asarray(i3[fam]["icp"][obs], float), np.asarray(ia[fam]["icp"][obs], float)
            xx = np.arange(len(cl)) + (k - 0.5) * 0.3
            ax[0, j].errorbar(xx - 0.05, v3[:, 0], v3[:, 1], fmt=mk, mfc="none", color=col, alpha=0.6, label=f"{fam}, phase 3 + 3e")
            ax[0, j].errorbar(xx + 0.05, va[:, 0], va[:, 1], fmt=mk, color=col, label=f"{fam}, + 3b")
            ax[1, j].plot(np.arange(len(cl)), v3[:, 1], mk + "--", mfc="none", color=col, label=f"{fam}, phase 3 + 3e")
            ax[1, j].plot(np.arange(len(cl)), va[:, 1], mk + "-", color=col, label=f"{fam}, + 3b")
            i0 = labs.index("0-10%")
            print(f"I_CP 0-10/40-60 {obs:15s} {fam}: {v3[i0, 0]:.2f}±{v3[i0, 1]:.2f} -> {va[i0, 0]:.2f}±{va[i0, 1]:.2f}")
        ax[0, j].axhline(1, color="k", lw=0.8); ax[0, j].set_ylim(0.4, 1.6); ax[0, j].set_title(obs, fontsize=9)
        ax[1, j].set_xticks(range(len(cl))); ax[1, j].set_xticklabels(labs, rotation=45, fontsize=8); ax[1, j].set_ylim(0, None)
    ax[0, 0].set_ylabel("$I_{CP}$ (class / 40-60 %)"); ax[1, 0].set_ylabel("statistical error"); ax[0, 0].legend(fontsize=7); ax[1, 0].legend(fontsize=7)
    fig.suptitle("Figure 5c. STAR's I_CP, what 3b changed: phase 3 + 3e (open) against + 3b (filled)", fontsize=11)
    fig.tight_layout(); plt.show()
    for fam in ("oo_vac", "oo_med"):
        print(f"I_AA {fam}: " + "  ".join(f"{k}: {i3[fam]['iaa'][k][0]:.3f}±{i3[fam]['iaa'][k][1]:.3f} -> {ia[fam]['iaa'][k][0]:.3f}±{ia[fam]['iaa'][k][1]:.3f}"
                                         for k in obs_))
    # R_CP 0-20/60-80 (MBD) errors for hadrons and jets, before / after
    def rcp_of(Rr, key, fam, a=(0, 20), b=(60, 80)):
        blk = Rr["raa"]["mbd"][key]; cl_ = [tuple(c) for c in blk["classes"]]
        y, e = np.asarray(blk[fam]["raa"]), np.asarray(blk[fam]["err"]); i_, j_ = cl_.index(a), cl_.index(b)
        r = y[i_] / y[j_]; return r, r * np.hypot(e[i_] / y[i_], e[j_] / y[j_])
    for key, edges in (("had", E["had"]), ("sph_jet", E["sph_jet"]), ("ch_jet_R0.2", E["ch_jet"]), ("ch_jet_R0.5", E["ch_jet"])):
        print(f"R_CP 0-20/60-80 (MBD) {key}, bins " + " ".join(f"{a:g}-{b:g}" for a, b in zip(edges[:-1], edges[1:])))
        for fam in ("oo_vac", "oo_med"):
            r3, e3 = rcp_of(A3E, key, fam); ra, ea = rcp_of(AALL, key, fam)
            print(f"   {fam} phase 3 + 3e: " + " ".join(f"{a:.2f}±{b:.2f}" for a, b in zip(r3, e3)))
            print(f"   {fam} + 3b        : " + " ".join(f"{a:.2f}±{b:.2f}" for a, b in zip(ra, ea)))
else:
    print("needs results/oo_analysis_rhic_phase3e.json and results/oo_analysis_rhic.json")

## 6. Systematics

- **Holes in hadron-level observables**: charged-hadron R_AA / R_CP and the h–h I_CP of the
  medium arm until `negative_hadrons.py` is run on the phase-3 medium files.
- **The 2/3 charged fraction** for the charged jets' hole subtraction (an approximation; the
  `*_raw` spectra bound the effect).
- **Matter's vacuum cutoff (QS)**: `pp_med`'s events without a medium against `pp_ref`'s, per window
  (the table below); quoted as the systematic on every medium/vacuum ratio.
- **MPI**: the frozen phase-2 arms with the old MPI and with MPI off (`g2_mpi_check.ipynb`); for
  the forward classes ±0.05 (MBD) / ±0.06 (EPD) on the vacuum R_CP.
- **Estimator**: EPD vs MBD vs mid-rapidity (Figures 1–3).
- **The flat UE** (`ue_trend_cv.ipynb`): mid-rapidity estimators carry the UE trend (−13 % at 11–20
  GeV, +6–19 % above, against STAR).
- **The energy-conservation trend** of the forward activity (`fwd_check.py`): the forward selection
  share is an upper estimate until constrained by STAR's p+Au event activity.

In [ ]:
qs = R.get("qs_check")
if qs:
    for key, rows in qs.items():
        print(key)
        for r in rows:
            print(f"  window {r['window'][0]:g}-{r['window'][1]:g}: no-medium events {r['n_med']} / {r['n_ref']}, "
                  f"total per event pp_med/pp_ref {r['total']:.3f} +- {r['total_err']:.3f}")
else:
    print("needs pp_med (3c)")

### Stitching: which p̂_T windows enter, and what is left out

The p̂_T windows are added with weights σ_k / N_k (× N_coll for O+O). A low window's rare entry in a
high bin carries a large weight, so a window enters a bin only where every compared sample has at least
10 raw entries there, **pooled over the classes**. That gives one choice per bin, the same for every
class, estimator and sample. A sparse class keeps the window and gets the larger error (from the sum
of squared weights). Until 2026-11-25 the sparsest class decided, which also dropped windows carrying
a real share of the yield: the 7–9 GeV window held about a quarter of the 7–30 GeV triggers, and the
5–7 GeV window about 40 % of the 5–6 GeV hadrons.

Below, per bin: the windows used; the largest share of a sample's yield in the windows left out
(should be ≲ 2 %); and the share of the lowest generated window. Above 2 % the bin is
**incomplete**, because p̂_T below 5 GeV is not generated and would contribute too. Incomplete bins
are shaded grey in Figures 1–3. The histograms are `hist` (boost-histogram) objects per
window × class × bin.

In [ ]:
EDGES_OF = {"had": E["had"], "sph_jet": E["sph_jet"], "ch_jet_R0.2": E["ch_jet"], "ch_jet_R0.5": E["ch_jet"]}
for key, edges in EDGES_OF.items():
    blk = R["raa"].get("mbd", {}).get(key)
    if not blk or "stitch" not in blk:
        continue
    st = blk["stitch"]
    print(f"{key}:")
    print("  bin            " + " ".join(f"{lo:g}-{hi:g}".rjust(7) for lo, hi in zip(edges[:-1], edges[1:])))
    print("  windows used   " + " ".join(f"{n:7d}" for n in blk["windows_used"]))
    for f, d in st["dropped_share"].items():
        print(f"  dropped {f:7s}" + " ".join(f"{x:7.3f}" for x in d))
    print("  lowest window  " + " ".join(f"{x:7.3f}" for x in st["lowest_window_share"]))
    print("  complete       " + " ".join(f"{'yes' if c else 'NO':>7s}" for c in st["complete"]))
ic = R.get("icp") or {}
if "stitch" in ic:
    st = ic["stitch"]
    print("STAR triggers 7-30 GeV: windows used", ", ".join(f"{lo:g}-{hi:g}" for lo, hi in ic["windows_used"]))
    for f, t in st["trigger_share_per_window"].items():
        print(f"  {f:7s} trigger share per window: " + " ".join(f"{w[0]:g}-{w[1]:g}:{x:.3f}" for w, x in zip(st["windows"], t))
              + f"  | dropped {st['dropped_share'][f]:.3f}")

## 6b. Hadronization: iColored vs Colorless on the same partons

**Why (Au+Au, 2026-11-26; `auau_jet_raa.ipynb`, Figs. 6–7).** In central Au+Au on our medium, the O+O chain
with Type 2, α_s 0.25 (B1) gives R_AA ≈ 0.5 for charged jets and hadrons, about 2× STAR / PHENIX and
JETSCAPE's own calculation (2204.01163 Fig. 14). JETSCAPE's chain on our medium (run C) reproduces Fig. 14,
so the medium is not the cause. B1's **own stored partons hadronized with Colorless** give run C's R_AA
(charged jets 0.51 → 0.23): on the same partons iColored makes about 2× the high-p_T hadrons of Colorless
in the medium, and the same as Colorless in vacuum. The energy given to the medium is the same; iColored's
colour connections of the recoils and the holes' partners (`colour_closure reconnect`) bring it back into
the leading hadrons and jets. iColored in a medium has no published calibration: 2407.17443 used it for
vacuum showers only (its energy-loss estimate used partonic jets and fragmentation functions), and the
recoil/hole colour flow exists in X-SCAPE since 2026-10-21.

**Why O+O may differ.** O+O is small: a leading parton reaches LBT (virtuality below Q_sw = 2 GeV) only after
≈ 1–2 fm, so most of its time in the medium is MATTER time, and it makes far fewer recoils and holes
(table below: ≈ 9 holes per jet event in central O+O against ≈ 80 in central Au+Au). The recapture, if it
comes from the recoil/hole closure, is correspondingly smaller, but so is O+O's whole quenching signal.

**Method.** Every O+O phase-3 sample stored its final partons, so `oo_med`, `oo_vac` and `pp_ref` were
re-hadronized offline with Colorless (`prod_OO_isr/rehadronize_colorless.py`, `refs/hadronize_pp19_icdecays.xml`;
the hard partons' status 1000 relabelled so Colorless takes them) and analysed with `oo_analysis_rhic.py
--out /data/OO-RHIC_phys/rehad_colorless --est-root /data/OO-RHIC_phys --tag colorless`: **every event keeps
the class of its original (iColored) estimators** and its bulk, so the comparison is the hadronization alone;
the holes are subtracted at hadron level (pstat −1 hadrons, JETSCAPE's way) instead of the parton-level
hole + partner subtraction. The Au+Au hadronization tests (B1's job with `--matter-only`, `--colour-closure
thermal`, `fake`; `auau_jet_raa.ipynb` Fig. 6c) locate the effect: iColored's string fragmentation of the
medium-modified shower, not the closure of the holes' colour lines.

In [ ]:
import glob, h5py, hdf5plugin
def medium_per_event(pattern, bmax=99.0, maxf=40):
    fs = [f for f in sorted(glob.glob(pattern)) if os.path.exists(f.replace("_hadrons_jet_frag.h5", ".json"))][:maxf]
    N = em = ho = rc = 0.0
    for p in fs:
        with h5py.File(p) as h:
            n = int(h.attrs["nunits_written"]); b = h["units/glauber_b"][:n]
            po = h["partons/offsets"][: n + 1]; par = h["partons/data"][: po[-1]]
        ok = (b < bmax)[np.repeat(np.arange(n), np.diff(po))]; P = par[ok]; N += (b < bmax).sum()
        em += P[P[:, 2] == 1, 3].sum() - P[P[:, 2] == -1, 3].sum(); ho += (P[:, 2] == -1).sum(); rc += (P[:, 2] == 1).sum()
    return N, em / max(N, 1), ho / max(N, 1), rc / max(N, 1)
print("p̂_T 23-30 GeV, per jet event: energy given to the medium (recoils - holes), holes, recoils")
for lab, pat, bm in (("O+O oo_med, b < 2.5 fm (central)", "/data/OO-RHIC_phys/oo_med_w23_30/*_hadrons_jet_frag.h5", 2.5),
                     ("O+O oo_med, all b < 10 fm", "/data/OO-RHIC_phys/oo_med_w23_30/*_hadrons_jet_frag.h5", 99.0),
                     ("Au+Au B1, b < 4.7 fm (0-10 %)", "/data/AuAu-RHIC_phys/auau_med_w23_30/*_hadrons_jet_frag.h5", 99.0)):
    N, e_, h_, r_ = medium_per_event(pat, bm)
    print(f"  {lab:34s} {int(N):6d} events | {e_:6.2f} GeV | {h_:6.1f} holes | {r_:6.1f} recoils")

In [ ]:
PC = Path("results/oo_analysis_rhic_colorless.json")
RC = json.loads(PC.read_text()) if PC.exists() else None
if RC is None:
    print("pending: results/oo_analysis_rhic_colorless.json (the Colorless re-hadronization of oo_med / oo_vac, then "
          "oo_analysis_rhic.py --out /data/OO-RHIC_phys/rehad_colorless --mb-root /data/OO-RHIC_phys "
          "--est-root /data/OO-RHIC_phys --tag colorless)")
else:
    def rcp_of(RR, est, key, fam, a, b):
        blk = RR["raa"][est][key]
        if fam not in blk:
            return None
        cl = [tuple(c) for c in blk["classes"]]; ia, ib = cl.index(a), cl.index(b)
        y, e = np.asarray(blk[fam]["raa"], float), np.asarray(blk[fam]["err"], float)
        with np.errstate(divide="ignore", invalid="ignore"):
            r = y[ia] / y[ib]
            return r, np.abs(r) * np.sqrt((e[ia] / y[ia]) ** 2 + (e[ib] / y[ib]) ** 2)
    def q_of(RR, est, key, c):
        blk = RR["raa"][est][key]
        a, ea = np.asarray(blk["oo_med"]["raa"], float)[c], np.asarray(blk["oo_med"]["err"], float)[c]
        b, eb = np.asarray(blk["oo_vac"]["raa"], float)[c], np.asarray(blk["oo_vac"]["err"], float)[c]
        with np.errstate(divide="ignore", invalid="ignore"):
            q = a / b
            return q, np.abs(q) * np.sqrt((ea / a) ** 2 + (eb / b) ** 2)
    SPD = (R.get("data") or {}).get("sphenix")
    HYB = {"iColored (as above)": (R, "o", "C3"), "Colorless, same partons": (RC, "D", "C2")}
    fig, ax = plt.subplots(1, 4, figsize=(20, 4))
    x, xe = xc(E["sph_jet"])
    # (a) sPHENIX R_CP 0-20 / 60-80 % (MBD)
    if SPD:
        rows = SPD["rcp_vs_60_80"]["red"]
        xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); wd = np.array([r_["hi"] - r_["lo"] for r_ in rows])
        ax[0].errorbar(xd, [r_["value"] for r_ in rows], [[r_["stat_lo"] for r_ in rows], [r_["stat_hi"] for r_ in rows]], wd / 2,
                       fmt="*", ms=8, color="k", label="sPHENIX prel.")
        rows = SPD["raa_0_100"]["black"]
        xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); wd = np.array([r_["hi"] - r_["lo"] for r_ in rows])
        ax[1].errorbar(xd, [r_["value"] for r_ in rows], [[r_["stat_lo"] for r_ in rows], [r_["stat_hi"] for r_ in rows]], wd / 2,
                       fmt="*", ms=8, color="k", label="sPHENIX prel.")
    for lab, (RR, mk, c_) in HYB.items():
        r = rcp_of(RR, "mbd", "sph_jet", "oo_med", (0, 20), (60, 80))
        if r is not None:
            ax[0].errorbar(x, r[0], r[1], xe, fmt=mk, ms=4, color=c_, label=f"oo_med, {lab}")
        blk = RR["raa"]["mbd"]["sph_jet"]
        if "oo_med" in blk and "raa_mb" in blk["oo_med"]:
            ax[1].errorbar(x, blk["oo_med"]["raa_mb"], blk["oo_med"]["err_mb"], xe, fmt=mk, ms=4, color=c_, label=f"oo_med, {lab}")
        # (c) Q_b 0-20 % (true geometry): quenching only, sPHENIX jets; (d) charged hadrons
        for k_, a_, ed in (("sph_jet", ax[2], E["sph_jet"]), ("had", ax[3], E["had"])):
            xx, xxe = xc(ed)
            q = q_of(RR, "b", k_, 0)
            a_.errorbar(xx, q[0], q[1], xxe, fmt=mk, ms=4, color=c_, label=lab)
    for a_, t in zip(ax, ("sPHENIX jets: R_CP 0-20 / 60-80 % (MBD)", "sPHENIX jets: R_AA 0-100 %",
                          "Q_b = R_AA(med)/R_AA(vac), 0-20 % by b: jets", "Q_b 0-20 % by b: charged hadrons |η| < 1")):
        a_.axhline(1, color="k", lw=0.8); a_.set_title(t, fontsize=9); a_.set_ylim(0.5, 1.3); a_.set_xlabel("$p_T$ [GeV]")
    ax[0].legend(fontsize=7); ax[2].legend(fontsize=7)
    fig.suptitle("Figure 6b. O+O, same events and partons: iColored (the analysis above) vs Colorless (model, not vetted)", fontsize=11)
    fig.tight_layout(); plt.show()
    for lab, (RR, *_ ) in HYB.items():
        for k_, lo, hi in (("sph_jet", 19, 60), ("had", 8, 20)):
            ed = E[k_]; s_ = (ed[:-1] >= lo - 1e-9) & (ed[1:] <= hi + 1e-9)
            q, e = q_of(RR, "b", k_, 0); w = 1 / np.where(e[s_] > 0, e[s_], np.inf) ** 2
            r = rcp_of(RR, "mbd", "sph_jet", "oo_med", (0, 20), (60, 80))
            msg = f"{lab:26s} {k_:8s} {lo}-{hi} GeV: Q_b(0-20 %) = {np.nansum(q[s_] * w) / np.sum(w):.3f}"
            if k_ == "sph_jet" and r is not None:
                ww = 1 / np.where(r[1][s_] > 0, r[1][s_], np.inf) ** 2
                msg += f" | R_CP 0-20/60-80 (MBD) = {np.nansum(r[0][s_] * ww) / np.sum(ww):.3f}"
            print(msg)

### Quenching per class, and the shower scheme

Q_b = R_AA(`oo_med`) / R_AA(`oo_vac`) per impact-parameter class, weighted over the p_T range, for both
hadronizations. **The shower scheme:** `oo_med` showers in the A+A scheme (Matter `in_vac 0`: down to QS =
0.9 GeV where there is no medium), `oo_vac` and `pp_ref` in the tune's scheme (`in_vac 1`: down to Q0 = 2
GeV). The `oo_med` events in which no medium formed (`has_medium` 0: no string, no MUSIC) against the same
events of `oo_vac` measure that scheme difference alone: no medium, no holes. Dividing it out of Q gives
the quenching.

In [ ]:
if RC is not None:
    def q_cls(RR, key, c, lo, hi):
        e = np.asarray(RR["edges"][key]); blk = RR["raa"]["b"][key]
        a, ea = np.asarray(blk["oo_med"]["raa"], float)[c], np.asarray(blk["oo_med"]["err"], float)[c]
        b, eb = np.asarray(blk["oo_vac"]["raa"], float)[c], np.asarray(blk["oo_vac"]["err"], float)[c]
        with np.errstate(divide="ignore", invalid="ignore"):
            q = a / b; eq = np.abs(q) * np.sqrt((ea / a) ** 2 + (eb / b) ** 2)
        s = (e[:-1] >= lo - 1e-9) & (e[1:] <= hi + 1e-9) & np.isfinite(q) & (eq > 0)
        w = 1 / eq[s] ** 2
        return np.sum(q[s] * w) / w.sum(), w.sum() ** -0.5
    def scheme(sfx, key, lo, hi):
        M = dict(np.load(f"results/oo_analysis_rhic_oo_med{sfx}.npz")); V = dict(np.load(f"results/oo_analysis_rhic_oo_vac{sfx}.npz"))
        e = np.asarray(R["edges"][key]); s = (e[:-1] >= lo) & (e[1:] <= hi)
        mm, mv = M["has_medium"] == 0, V["has_medium"] == 0
        ym = (M["w"][mm][:, None] * M[key][mm][:, s]).sum() / M["w"][mm].sum()
        yv = (V["w"][mv][:, None] * V[key][mv][:, s]).sum() / V["w"][mv].sum()
        return ym / yv, int(mm.sum())
    RNG = (("had", 4, 8), ("had", 8, 20), ("sph_jet", 19, 60))
    SCH = {}
    print("shower scheme (oo_med / oo_vac in the events without a medium):")
    for nm, sfx in (("iColored", ""), ("Colorless", "_colorless")):
        for key, lo, hi in RNG:
            SCH[(nm, key, lo)], nev = scheme(sfx, key, lo, hi)
        print(f"  {nm:9s} " + "  ".join(f"{k} {lo}-{hi}: {SCH[(nm, k, lo)]:.3f}" for k, lo, hi in RNG) + f"   ({nev} events)")
    cl = R["raa"]["b"]["had"]["classes"]
    print("\nQ_b per class (raw | scheme divided out):")
    for key, lo, hi in RNG:
        print(f"  {key} {lo}-{hi} GeV")
        for c, (a_, b_) in enumerate(cl):
            row = []
            for nm, RR in (("iColored", R), ("Colorless", RC)):
                q, e = q_cls(RR, key, c, lo, hi); f = SCH[(nm, key, lo)]
                row.append(f"{nm} {q:.3f}±{e:.3f} | {q / f:.3f}")
            print(f"    {a_:>3d}-{b_:<3d}%  " + "    ".join(row))

**Reading (6b), 2026-11-26** (model results, not vetted):

1. **The vacuum arm does not depend on the hadronization**: `oo_vac` against `pp_ref` is ≈ 1 by b with
   either (the selection is the same).
2. **The medium arm does, strongly.** Central O+O (0–20 % by b), Q_b: charged hadrons 8–20 GeV
   1.16 ± 0.08 (iColored) vs 0.63 ± 0.05 (Colorless); sPHENIX jets 19–60 GeV 0.90 ± 0.02 vs 0.80 ± 0.02.
   iColored even *enhances* the high-p_T hadrons in the medium.
3. **The shower scheme biases Colorless more.** In the events without a medium, `oo_med`/`oo_vac` is
   0.90–0.96 (hadrons) and 0.95 (jets) with iColored, 0.82 and 0.93 with Colorless: the A+A scheme's extra
   splittings below 2 GeV soften Colorless's leading hadrons by ≈ 18 %. Divided out, central O+O quenching:
   **Colorless hadrons ≈ 0.77, jets ≈ 0.86; iColored hadrons ≈ 1.3, jets ≈ 0.95.** Peripheral classes are
   then ≈ 1 for both, as they should be. (The same scheme mismatch enters the Au+Au comparison of
   `auau_jet_raa.ipynb`: B1's partons with Colorless against `pp_ref` Colorless; it lowers that R_AA by
   ≈ 7 % for jets and ≈ 18 % for hadrons, and does not change the factor ≈ 2 there.)
4. **So in O+O, too, iColored hides most of the quenching** that Colorless sees on the same partons: with
   the scheme divided out, central jets ≈ 0.86 (Colorless) vs ≈ 0.95 (iColored), hadrons ≈ 0.77 vs ≈ 1.3.
   Against sPHENIX (Fig. 6b): the jet R_AA 0–100 % is 0.81–0.84 with Colorless at 13–40 GeV, on the
   preliminary 0.79–0.82, but ≈ 7 % of that is the scheme (≈ 0.87–0.90 without it); iColored 0.94–0.98.
   R_CP 0–20 / 60–80 % (MBD; the scheme largely cancels) 19–60 GeV: 0.74 (Colorless) and 0.79 (iColored),
   against sPHENIX's ≈ 0.62–0.69 at 20–43 GeV: Colorless moves towards the data, but does not reach it.
5. **Not the colour fixes, not the closure.** The duplicate-tag fix changed iColored's jet hadrons by 5–17 %
   and the closures by ≲ 1 % (`prod_OO_isr/hadro_compare`, 2026-10-23, O+O without LBT). In Au+Au
   (`auau_jet_raa.ipynb`, Fig. 6c): MATTER alone (T1, no recoils, holes or closure) gives iColored ≈ 1.4×
   Colorless's high-p_T hadrons from the same partons; with LBT ≈ 2×, and the same with every closure
   (`reconnect` B1, `thermal` T2, `fake` T3); vacuum ≈ 1.06×. It is iColored's string fragmentation of a
   medium-modified shower itself. O+O's showers are mostly MATTER showers, so the T1 part applies directly.
6. **What it needs:** a reference in the A+A scheme (phase 4: `oo_vac` and `pp_ref` with Matter `in_vac 0`),
   which removes the scheme term for any hadronization; and a decision on the hadronization of medium-modified
   showers before the energy-loss parameters are calibrated (Au+Au) and O+O is compared with data.

## 7. Reading (2026-11-26: phase 3 + 3b + 3e; 3c and phase 4 not run)

Model results, not vetted. sPHENIX jets R = 0.4, MBD classes, 19–60 GeV (weighted over the bins).

1. **The selection dominates the measured R_CP.** The vacuum arm (selection only) gives R_CP 0–20 / 60–80 % =
   0.89 ± 0.03 (19–36 GeV) and 0.80 ± 0.03 (36–60 GeV) (Figs. 1, 3); the jet events' forward signal falls with p̂_T
   because the hard scattering takes energy from the beam remnants. The medium arm (selection × quenching) gives
   0.87 ± 0.04 and 0.74 ± 0.03: the quenching adds only a few percent to the R_CP.
2. **The quenching at the true geometry is small**: Q_b (0–20 % by b) ≈ 0.90 for the jets with iColored, ≈ 0.95
   once the shower-scheme mismatch is divided out (6b); the hadrons show no suppression with iColored.
3. **Against sPHENIX (preliminary):** R_CP 0–20 / 60–80 % 0.79 ± 0.02 (model) vs ≈ 0.62–0.69 (data, 20–43 GeV);
   R_AA 0–100 % 0.95 ± 0.01 vs 0.79–0.82. The model is above the data in both, by ≈ 0.1–0.15.
4. **The hadronization is a first-order systematic (6b).** On the same partons Colorless gives R_CP 0.74 ± 0.02
   and R_AA 0.87 ± 0.01 (≈ 0.93 without the scheme term), and Q_b ≈ 0.86 (jets) and ≈ 0.77 (hadrons): it moves the
   model towards the data, but does not reach it. In Au+Au the same chain with iColored under-quenches by ≈ 2×
   (`auau_jet_raa.ipynb`), whereas JETSCAPE's chain on our medium describes STAR and PHENIX: the O+O energy loss
   is not calibrated yet, and its O+O quenching is a lower estimate.
5. **Before the O+O comparison is final:** decide the hadronization of medium-modified showers and calibrate the
   energy loss with it in Au+Au; a reference in the A+A shower scheme (phase 4); then rerun `oo_med` (and 3c,
   `pp_med`, if wanted) with the chosen setting. The selection share (point 1) does not depend on the
   hadronization (the vacuum arm is the same with both) and stands.